# Project 01 - Data Cleaning on an E-commerce Dataset

Before running: put the raw excel file in the same folder as this notebook, named `raw_dataset.xlsx`. If your file has a different name, just change it in the next cell.

In [1]:
import pandas as pd
import numpy as np

file_name = 'raw_dataset.xlsx'  # change this if your file is named differently

df = pd.read_excel(file_name)
df.shape

(61, 17)

## Quick look at the data

In [2]:
df.head()

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
0,1001,Reza,F,19.0,Karaj,Alborz,2025-02-19,VIP,17,121.53,2066.01,16,Card,Android,Yes,3,5
1,1002,Sina,M,53.0,Tehran,Tehran,2022-08-19,Gold,12,326.47,3917.64,3,Card,Web,No,5,3
2,1003,Parsa,F,31.0,Shiraz,Fars,2023-06-20,Gold,21,59.46,1248.66,22,Online Wallet,iPhone,Yes,6,1
3,1004,Sina,F,58.0,Mashhad,Khorasan,2021-11-08,Gold,23,266.15,6121.45,40,Card,Android,No,4,4
4,1005,Kimia,M,28.0,Isfahan,Isfahan,2021-10-21,Silver,23,169.54,3899.42,273,Online Wallet,Android,Yes,7,4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         61 non-null     int64  
 1   first_name          61 non-null     object 
 2   gender              61 non-null     object 
 3   age                 60 non-null     float64
 4   city                61 non-null     object 
 5   province            61 non-null     object 
 6   signup_date         61 non-null     object 
 7   membership_tier     61 non-null     object 
 8   purchase_count      61 non-null     int64  
 9   avg_order_value     61 non-null     float64
 10  total_spending      60 non-null     float64
 11  last_purchase_days  61 non-null     int64  
 12  payment_method      61 non-null     object 
 13  device              61 non-null     object 
 14  discount_used       61 non-null     object 
 15  returned_items      61 non-null     int64  
 16  satisfacti

In [4]:
df.describe()

,customer_id,age,purchase_count,avg_order_value,total_spending,last_purchase_days,returned_items,satisfaction_score
count,61.000000,60.000000,61.000000,61.000000,60.000000,61.000000,61.000000,61.00000
mean,1030.229508,45.200000,17.606557,211.435738,3749.667333,196.934426,4.196721,3.00000
std,17.446483,19.167239,10.214824,130.113726,4262.248522,97.892436,2.663454,1.42595
min,1001.000000,19.000000,0.000000,27.630000,0.000000,3.000000,0.000000,1.00000
25%,1015.000000,31.750000,11.000000,108.190000,1167.035000,132.000000,2.000000,2.00000
50%,1030.000000,45.000000,17.000000,161.330000,2188.865000,201.000000,4.000000,3.00000
75%,1045.000000,58.000000,26.000000,323.320000,4692.150000,273.000000,7.000000,4.00000
max,1060.000000,145.000000,35.000000,449.810000,25000.000000,365.000000,8.000000,5.00000


---
# 1. Missing Values

In [5]:
df.isna().sum()

customer_id           0
first_name            0
gender                0
age                   1
city                  0
province              0
signup_date           0
membership_tier       0
purchase_count        0
avg_order_value       0
total_spending        1
last_purchase_days    0
payment_method        0
device                0
discount_used         0
returned_items        0
satisfaction_score    0
dtype: int64

For numbers I'm filling with the median (safer than the average if there are outliers). For text columns I'll use the most common value. Not dropping rows - a customer with one missing field still has 16 other useful fields.

In [6]:
for col in df.select_dtypes(include=[np.number]).columns:
    missing = df[col].isna().sum()
    if missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(col, '- filled', missing, 'value(s) with median:', median_val)

for col in df.select_dtypes(include=['object']).columns:
    missing = df[col].isna().sum()
    if missing > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(col, '- filled', missing, 'value(s) with most common value:', mode_val)

age - filled 1 value(s) with median: 45.0
total_spending - filled 1 value(s) with median: 2188.865


In [7]:
df.isna().sum().sum()  # should be 0 now

0

---
# 2. Duplicate Records

customer_id should never repeat, so checking that first.

In [8]:
df.duplicated(subset=['customer_id']).sum()

1

In [9]:
df[df.duplicated(subset=['customer_id'], keep=False)].sort_values('customer_id')

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
13,1014,Reza,M,35.0,Tabriz,East Azerbaijan,2022-08-26,VIP,31,108.19,3353.89,97,Card,Android,No,5,4
60,1014,Reza,M,35.0,Tabriz,East Azerbaijan,2022-08-26,VIP,31,108.19,3353.89,97,Card,Android,No,5,4


In [10]:
rows_before = len(df)
df = df.drop_duplicates(subset=['customer_id'], keep='first').reset_index(drop=True)
print(rows_before, '->', len(df))

61 -> 60


---
# 3. Incorrect Data Types

In [11]:
df.dtypes

customer_id             int64
first_name             object
gender                 object
age                   float64
city                   object
province               object
signup_date            object
membership_tier        object
purchase_count          int64
avg_order_value       float64
total_spending        float64
last_purchase_days      int64
payment_method         object
device                 object
discount_used          object
returned_items          int64
satisfaction_score      int64
dtype: object

In [12]:
int_cols = ['customer_id', 'purchase_count', 'last_purchase_days', 'returned_items', 'satisfaction_score']
for col in int_cols:
    df[col] = df[col].astype(int)

float_cols = ['avg_order_value', 'total_spending']
for col in float_cols:
    df[col] = df[col].round(2).astype(float)

df['discount_used'] = df['discount_used'].map({'Yes': True, 'No': False, 'yes': True, 'no': False})

df.dtypes

customer_id             int32
first_name             object
gender                 object
age                   float64
city                   object
province               object
signup_date            object
membership_tier        object
purchase_count          int32
avg_order_value       float64
total_spending        float64
last_purchase_days      int32
payment_method         object
device                 object
discount_used            bool
returned_items          int32
satisfaction_score      int32
dtype: object

---
# 4. Invalid Values

Checking age since that's an easy one to spot problems in (nobody is 150 years old).

In [13]:
df['age'].describe()

count     60.000000
mean      45.366667
std       19.120463
min       19.000000
25%       31.750000
50%       45.000000
75%       58.000000
max      145.000000
Name: age, dtype: float64

In [14]:
weird_ages = df[(df['age'] < 10) | (df['age'] > 100)]
weird_ages[['customer_id', 'age']]

,customer_id,age
9,1010,145.0


In [15]:
if len(weird_ages) > 0:
    median_age = df['age'].median()
    mask = (df['age'] < 10) | (df['age'] > 100)
    df.loc[mask, 'age'] = median_age
    df['age'] = df['age'].astype(int)
    print('fixed', mask.sum(), 'unrealistic age value(s), replaced with median:', median_age)

fixed 1 unrealistic age value(s), replaced with median: 45.0


In [16]:
# checking nothing is negative where it shouldn't be
for col in ['purchase_count', 'avg_order_value', 'total_spending', 'returned_items']:
    neg = (df[col] < 0).sum()
    print(col, '- negative values:', neg)

purchase_count - negative values: 0
avg_order_value - negative values: 0
total_spending - negative values: 0
returned_items - negative values: 0


In [17]:
# satisfaction_score should only be 1-5
out_of_range = df[(df['satisfaction_score'] < 1) | (df['satisfaction_score'] > 5)]
print('out of range satisfaction scores:', len(out_of_range))

out of range satisfaction scores: 0


---
# 5. Outliers

total_spending should roughly equal purchase_count times avg_order_value. If it's way off, something's wrong with one of these three numbers.

In [18]:
df['expected'] = (df['purchase_count'] * df['avg_order_value']).round(2)
df['gap'] = (df['total_spending'] - df['expected']).abs()

mismatch = df[df['gap'] > df['expected'] * 0.5]
mismatch[['customer_id', 'purchase_count', 'avg_order_value', 'total_spending', 'expected']]

,customer_id,purchase_count,avg_order_value,total_spending,expected
29,1030,26,156.89,25000.00,4079.14
39,1040,34,37.66,2188.86,1280.44


In [19]:
# if the gap is more than 5x, it's probably just a typo (extra digit) - safe to fix
big_gap = df['gap'] > 5 * df['expected']
print('fixing', big_gap.sum(), 'row(s) with an obvious data entry error')
df.loc[big_gap, 'total_spending'] = df.loc[big_gap, 'expected']

# smaller mismatches I'm not touching - could be any of the 3 columns that's wrong,
# no way to know which one without more info, so better to leave it and flag it
smaller_gap = (df['gap'] > df['expected'] * 0.5) & (~big_gap)
print('flagging', smaller_gap.sum(), 'row(s) as inconsistent but leaving them as-is:')
print(df.loc[smaller_gap, 'customer_id'].tolist())

df = df.drop(columns=['expected', 'gap'])

fixing 1 row(s) with an obvious data entry error
flagging 1 row(s) as inconsistent but leaving them as-is:
[1040]


---
# 6. Inconsistent Naming

Checking every text column for capitalization issues - "vip", "VIP" and "Vip" should all be one category, not three.

In [20]:
text_cols = [c for c in df.select_dtypes(include=['object']).columns if c != 'customer_id']

for col in text_cols:
    print(col, '->', sorted(df[col].astype(str).unique()))

first_name -> ['Ali', 'Amir', 'Arash', 'Kimia', 'Maryam', 'Mina', 'Neda', 'Parsa', 'Reza', 'Sara', 'Sina', 'Zahra']
gender -> ['F', 'M']
city -> ['Ahvaz', 'Isfahan', 'Karaj', 'Mashhad', 'Rasht', 'Shiraz', 'Tabriz', 'Tehran']
province -> ['Alborz', 'East Azerbaijan', 'Fars', 'Gilan', 'Isfahan', 'Khorasan', 'Khuzestan', 'Tehran']
signup_date -> ['2021-02-23', '2021-02-24', '2021-02-25', '2021-03-26', '2021-04-27', '2021-05-14', '2021-06-25', '2021-07-24', '2021-07-27', '2021-10-08', '2021-10-14', '2021-10-21', '2021-11-02', '2021-11-08', '2021-12-16', '2021-12-25', '2022-01-02', '2022-01-26', '2022-02-14', '2022-03-18', '2022-04-24', '2022-05-06', '2022-05-15', '2022-06-10', '2022-06-25', '2022-07-01', '2022-08-05', '2022-08-07', '2022-08-11', '2022-08-19', '2022-08-26', '2023-01-02', '2023-06-06', '2023-06-20', '2023-07-03', '2023-07-22', '2023-08-04', '2023-09-05', '2024-01-12', '2024-02-20', '2024-05-02', '2024-06-09', '2024-06-11', '2024-06-25', '2024-07-27', '2024-08-15', '2024-09-0

In [21]:
# title case everywhere (First Letter Big, rest small) - consistent and readable
for col in text_cols:
    df[col] = df[col].astype(str).str.title()

for col in text_cols:
    print(col, '->', sorted(df[col].unique()))

first_name -> ['Ali', 'Amir', 'Arash', 'Kimia', 'Maryam', 'Mina', 'Neda', 'Parsa', 'Reza', 'Sara', 'Sina', 'Zahra']
gender -> ['F', 'M']
city -> ['Ahvaz', 'Isfahan', 'Karaj', 'Mashhad', 'Rasht', 'Shiraz', 'Tabriz', 'Tehran']
province -> ['Alborz', 'East Azerbaijan', 'Fars', 'Gilan', 'Isfahan', 'Khorasan', 'Khuzestan', 'Tehran']
signup_date -> ['2021-02-23', '2021-02-24', '2021-02-25', '2021-03-26', '2021-04-27', '2021-05-14', '2021-06-25', '2021-07-24', '2021-07-27', '2021-10-08', '2021-10-14', '2021-10-21', '2021-11-02', '2021-11-08', '2021-12-16', '2021-12-25', '2022-01-02', '2022-01-26', '2022-02-14', '2022-03-18', '2022-04-24', '2022-05-06', '2022-05-15', '2022-06-10', '2022-06-25', '2022-07-01', '2022-08-05', '2022-08-07', '2022-08-11', '2022-08-19', '2022-08-26', '2023-01-02', '2023-06-06', '2023-06-20', '2023-07-03', '2023-07-22', '2023-08-04', '2023-09-05', '2024-01-12', '2024-02-20', '2024-05-02', '2024-06-09', '2024-06-11', '2024-06-25', '2024-07-27', '2024-08-15', '2024-09-0

gender is only one letter (M/F) so title case leaves it fine as-is. Checking membership_tier specifically since VIP should stay as one consistent value:

In [22]:
df['membership_tier'].unique()

array(['Vip', 'Gold', 'Silver', 'Bronze'], dtype=object)

### Gender vs first name

In Persian, these first names aren't shared between genders the way "Jordan" or "Alex" would be in English - each one is clearly male-only or female-only. So if a name shows up with both M and F in this data, gender is the one that's wrong, not the name.

In [23]:
for name in sorted(df['first_name'].unique()):
    counts = df[df['first_name'] == name]['gender'].value_counts()
    print(name, '-', counts.to_dict())

Ali - {'F': 5, 'M': 1}
Amir - {'F': 4, 'M': 2}
Arash - {'F': 2, 'M': 2}
Kimia - {'M': 5, 'F': 2}
Maryam - {'M': 3, 'F': 1}
Mina - {'M': 2, 'F': 1}
Neda - {'F': 6, 'M': 2}
Parsa - {'F': 3, 'M': 1}
Reza - {'F': 5, 'M': 2}
Sara - {'F': 1}
Sina - {'F': 5, 'M': 3}
Zahra - {'M': 2}


double check this list matches your own data before running the next cell, in case your file has different names.

In [24]:
male_names = ['Reza', 'Amir', 'Parsa', 'Sina', 'Arash', 'Ali']
female_names = ['Kimia', 'Neda', 'Maryam', 'Sara', 'Mina', 'Zahra']

for name in male_names:
    mask = (df['first_name'] == name) & (df['gender'] != 'M')
    if mask.sum() > 0:
        print('fixing', mask.sum(), 'row(s) -', name, 'should be M')
        df.loc[mask, 'gender'] = 'M'

for name in female_names:
    mask = (df['first_name'] == name) & (df['gender'] != 'F')
    if mask.sum() > 0:
        print('fixing', mask.sum(), 'row(s) -', name, 'should be F')
        df.loc[mask, 'gender'] = 'F'

fixing 5 row(s) - Reza should be M
fixing 4 row(s) - Amir should be M
fixing 3 row(s) - Parsa should be M
fixing 5 row(s) - Sina should be M
fixing 2 row(s) - Arash should be M
fixing 5 row(s) - Ali should be M
fixing 5 row(s) - Kimia should be F
fixing 2 row(s) - Neda should be F
fixing 3 row(s) - Maryam should be F
fixing 2 row(s) - Mina should be F
fixing 2 row(s) - Zahra should be F


In [25]:
# double check - every name should now show only one gender
for name in sorted(df['first_name'].unique()):
    counts = df[df['first_name'] == name]['gender'].value_counts()
    print(name, '-', counts.to_dict())

Ali - {'M': 6}
Amir - {'M': 6}
Arash - {'M': 4}
Kimia - {'F': 7}
Maryam - {'F': 4}
Mina - {'F': 3}
Neda - {'F': 8}
Parsa - {'M': 4}
Reza - {'M': 7}
Sara - {'F': 1}
Sina - {'M': 8}
Zahra - {'F': 2}


### Outdated province name

Checking if city and province actually line up consistently first - if one city shows up under two different province names, that's a real data problem. If each city consistently maps to one province, the data itself is internally fine, and "Khorasan" is more of an outdated/incomplete name issue than a data error (Iran split the old Khorasan province into three - Razavi, North, South - back in 2004, and Mashhad is in Khorasan Razavi).

In [26]:
df.groupby('city')['province'].unique()

city
Ahvaz            [Khuzestan]
Isfahan            [Isfahan]
Karaj               [Alborz]
Mashhad           [Khorasan]
Rasht                [Gilan]
Shiraz                [Fars]
Tabriz     [East Azerbaijan]
Tehran              [Tehran]
Name: province, dtype: object

In [27]:
# each city maps to exactly one province here, so this isn't inconsistent data -
# it's just an old/incomplete province name. updating it to be accurate.
df['province'] = df['province'].replace({'Khorasan': 'Khorasan Razavi'})
df['province'].unique()

array(['Alborz', 'Tehran', 'Fars', 'Khorasan Razavi', 'Isfahan', 'Gilan',
       'East Azerbaijan', 'Khuzestan'], dtype=object)

---
# 7. Typos

Checking for anything that looks like a misspelled category (close to a real value but not quite).

In [28]:
for col in text_cols:
    print(col, '->', sorted(df[col].unique()))

first_name -> ['Ali', 'Amir', 'Arash', 'Kimia', 'Maryam', 'Mina', 'Neda', 'Parsa', 'Reza', 'Sara', 'Sina', 'Zahra']
gender -> ['F', 'M']
city -> ['Ahvaz', 'Isfahan', 'Karaj', 'Mashhad', 'Rasht', 'Shiraz', 'Tabriz', 'Tehran']
province -> ['Alborz', 'East Azerbaijan', 'Fars', 'Gilan', 'Isfahan', 'Khorasan Razavi', 'Khuzestan', 'Tehran']
signup_date -> ['2021-02-23', '2021-02-24', '2021-02-25', '2021-03-26', '2021-04-27', '2021-05-14', '2021-06-25', '2021-07-24', '2021-07-27', '2021-10-08', '2021-10-14', '2021-10-21', '2021-11-02', '2021-11-08', '2021-12-16', '2021-12-25', '2022-01-02', '2022-01-26', '2022-02-14', '2022-03-18', '2022-04-24', '2022-05-06', '2022-05-15', '2022-06-10', '2022-06-25', '2022-07-01', '2022-08-05', '2022-08-07', '2022-08-11', '2022-08-19', '2022-08-26', '2023-01-02', '2023-06-06', '2023-06-20', '2023-07-03', '2023-07-22', '2023-08-04', '2023-09-05', '2024-01-12', '2024-02-20', '2024-05-02', '2024-06-09', '2024-06-11', '2024-06-25', '2024-07-27', '2024-08-15', '20

went through the printed lists above - nothing looks like a misspelling (no 'Cardd' or 'Androidd' type issues). If you spot one in your real file, fix it here with `.replace({'wrong': 'correct'})`.

---
# 8. Extra Whitespace

Already partly handled by `.str.title()` above, since Python capitalization would leave leading/trailing spaces untouched otherwise. Double checking explicitly here.

In [29]:
for col in text_cols:
    has_space = (df[col] != df[col].str.strip()).sum()
    if has_space > 0:
        print(col, '-', has_space, 'value(s) had extra whitespace')
        df[col] = df[col].str.strip()
    else:
        print(col, '- no extra whitespace found')

first_name - no extra whitespace found
gender - no extra whitespace found
city - no extra whitespace found
province - no extra whitespace found
signup_date - no extra whitespace found
membership_tier - no extra whitespace found
payment_method - no extra whitespace found
device - no extra whitespace found


---
# 9. Inconsistent Date Formats

In [30]:
df['signup_date'].dtype

dtype('O')

In [31]:
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['signup_date'].isna().sum()  # any date that failed to parse

0

In [32]:
# dates that are somehow in the future don't make sense for a signup date
future = df[df['signup_date'] > pd.Timestamp.today()]
future[['customer_id', 'signup_date']]

,customer_id,signup_date


---
# 10. Logical Inconsistency

Checking that different columns actually agree with each other.

In [33]:
# a customer with 0 purchases shouldn't have money spent
weird = df[(df['purchase_count'] == 0) & (df['total_spending'] > 0)]
weird[['customer_id', 'purchase_count', 'total_spending']]

,customer_id,purchase_count,total_spending


In [34]:
# can't return more items than you bought
bad_returns = df[df['returned_items'] > df['purchase_count']]
print('rows where returned_items > purchase_count:', len(bad_returns))
bad_returns[['customer_id', 'purchase_count', 'returned_items']]

rows where returned_items > purchase_count: 6


,customer_id,purchase_count,returned_items
7,1008,3,8
14,1015,3,8
23,1024,0,2
28,1029,2,4
36,1037,1,7
55,1056,1,4


In [35]:
# not enough info to know if purchase_count or returned_items is the wrong number here,
# so flagging these instead of guessing which one to change
if len(bad_returns) > 0:
    print('flagged', len(bad_returns), 'row(s) for manual review:', bad_returns['customer_id'].tolist())
    print('left unchanged - could be either column that has the actual error')

flagged 6 row(s) for manual review: [1008, 1015, 1024, 1029, 1037, 1056]
left unchanged - could be either column that has the actual error


---
# Questions This Cleaned Dataset Should Be Able to Answer

Quick sanity check before calling this "analysis-ready" - actually trying to answer the questions this dataset is supposed to support, using the cleaned `df`.

In [36]:
# which cities generate the most revenue
df.groupby('city')['total_spending'].sum().sort_values(ascending=False)

city
Mashhad    43197.58
Tabriz     36192.49
Karaj      26189.81
Ahvaz      23384.78
Isfahan    21975.32
Shiraz     20715.92
Tehran     16069.71
Rasht      15168.54
Name: total_spending, dtype: float64

In [37]:
# which customers bought the most (by purchase count)
df.sort_values('purchase_count', ascending=False)[['customer_id', 'first_name', 'purchase_count']].head(5)

,customer_id,first_name,purchase_count
5,1006,Amir,35
39,1040,Amir,34
8,1009,Kimia,34
43,1044,Mina,33
22,1023,Ali,31


In [38]:
# average order value across all customers
df['avg_order_value'].mean()

213.1565

In [39]:
# most used payment method
df['payment_method'].value_counts()

payment_method
Online Wallet    23
Cash             19
Card             18
Name: count, dtype: int64

In [46]:
# effect of discounts
discount_comparison = df.groupby('discount_used').agg(
    customers=('customer_id', 'count'),
    avg_total_spending=('total_spending', 'mean'),
    avg_purchase_count=('purchase_count', 'mean'),
    avg_order_value=('avg_order_value', 'mean'),
    avg_satisfaction=('satisfaction_score', 'mean'),
    avg_returned_items=('returned_items', 'mean')
).round(2)

discount_comparison

,customers,avg_total_spending,avg_purchase_count,avg_order_value,avg_satisfaction,avg_returned_items
discount_used,,,,,,
False,34,3736.74,17.29,220.70,2.76,4.26
True,26,2917.11,17.50,203.29,3.27,4.08


**Busiest time period?** This dataset only has `signup_date`, not an actual order/purchase date, so this can only be answered indirectly (when customers joined, not when they bought). Treat this as a rough signal, not a real sales trend.

In [41]:
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['signup_date'].dt.to_period('M').value_counts().sort_index()

signup_date
2021-02    3
2021-03    1
2021-04    1
2021-05    1
2021-06    1
2021-07    2
2021-10    3
2021-11    3
2021-12    2
2022-01    2
2022-02    1
2022-03    1
2022-04    1
2022-05    2
2022-06    2
2022-07    1
2022-08    5
2023-01    2
2023-06    2
2023-07    2
2023-08    1
2023-09    1
2024-01    1
2024-02    1
2024-05    1
2024-06    3
2024-07    1
2024-08    1
2024-09    2
2024-10    2
2024-11    1
2024-12    1
2025-01    2
2025-02    1
2025-03    1
2025-07    1
2025-09    1
Freq: M, Name: count, dtype: int64

**Is there anything in the data that could lead to a wrong conclusion?** Yes, worth keeping in mind for later analysis:
- The 6 rows with `returned_items > purchase_count` and the 1 flagged `total_spending` mismatch (customer 1040) are still slightly off - they were flagged, not corrected, so any total that includes them carries a small amount of uncertainty.
- `signup_date` is not the same as purchase date, so anything about "when sales happened" is really just "when customers joined," which is a weaker signal.
- No product-level data exists, so revenue/city/customer questions can be answered, but "what sells" cannot.
- The dataset is fairly small (60 customers) - patterns here are more directional than statistically solid, especially for smaller subgroups like specific cities or membership tiers.

In [42]:
# average spending by membership tier - are higher tiers actually worth more?
df.groupby('membership_tier')['total_spending'].mean().sort_values(ascending=False)

membership_tier
Silver    4179.926250
Gold      3955.431579
Bronze    3217.123889
Vip       2426.220667
Name: total_spending, dtype: float64

---
# Final Check & Save

In [44]:
print('rows:', len(df))
print('missing values left:', df.isna().sum().sum())
print('duplicate customer_id left:', df.duplicated(subset='customer_id').sum())

rows: 60
missing values left: 0
duplicate customer_id left: 0


In [45]:
df['signup_date'] = df['signup_date'].dt.strftime('%Y-%m-%d')
df.to_excel('cleaned_dataset.xlsx', index=False)
df.to_csv('cleaned_dataset.csv', index=False)
print('saved')

saved
